# NWB EDA — Junction Disaggregation Problem

Multiple NWB junction nodes often describe the same physical intersection,
primarily because dual carriageways (FOW=2) are digitised as separate linestrings.

This notebook explores the problem and evaluates candidate solutions:

| # | Approach | Key attribute(s) | Verdict |
|---|---|---|---|
| A | FOW=2 flag — scope which junctions are affected | `FOW` | **Used** — scope filter, combined with D |
| B | RIJRICHTNG — confirm forced direction as dual-carriageway signal | `RIJRICHTNG` | **Not used** — unreliable overlap with FOW=2 |
| C | RPE_CODE — pair L/R carriageways of the same road | `RPE_CODE`, `WEGNUMMER` | **Not used** — sparsely populated for municipal roads |
| D | Proximity clustering — group junctions within a distance threshold | geometry | **Used** — core merge signal, two thresholds (FOW=2: 30 m, bayonet: 6 m) |
| E | Graph topology contraction — collapse nodes linked by short edges | graph | **Not used** — no clean length threshold in the data |

**Chosen approach:** A + D combined. FOW=2-connected junction pairs within 30 m are merged; all junction pairs within 6 m (bayonet signal) are merged regardless of FOW. See `exp_eda_nwb_bayonet.ipynb` for threshold selection and `01_merge_junctions.ipynb` for the implementation.

## 0. Setup

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from shapely.geometry import Point

# --- Paths ---
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"

# Preprocessed Rotterdam-only files (run scripts/preprocess_nwb.py once to generate these).
# Using these avoids loading the large national Wegvakken.gpkg every session.
WEGVAKKEN_ROT_PATH = os.path.join(PROJECT_DIR, "data", "processed", "wegvakken_rotterdam.gpkg")
WEGVAKKEN_BST_PATH = os.path.join(PROJECT_DIR, "data", "processed", "wegvakken_rotterdam_bst.gpkg")

# Fallback: national file (only used if the processed files don't exist yet)
WEGVAKKEN_NL_PATH  = os.path.join(PROJECT_DIR, "data", "raw", "Wegvakken.gpkg")

INTERSECTIONS_PATH = os.path.join(PROJECT_DIR, "data", "processed", "intersections.gpkg")

# --- Constants ---
CRS_RD = "EPSG:28992"   # RD New â€” all NWB data is in this projection

# Road types (BST_CODE) to include â€” only "normal" roads.
# Excludes: NRB (side carriageways), AFR (off-ramps), OPR (on-ramps), FP, parking.
INCLUDE_BST = {"RB", "ERF", "HR"}

# Road manager filter: only gemeente-managed roads (excludes Rijk highways, Provincie).
WEGBEHSRT_GEMEENTE = "G"

## 1. Load data

We load three levels of filtering to separate concerns:
- `wegvakken_rot` â€” all Rotterdam segments (no filter), for full context
- `wegvakken_gemeente` â€” WEGBEHSRT='G' only (gemeente-managed, no BST filter) â€” used for FOW analysis
- `wegvakken` â€” WEGBEHSRT='G' + BST_CODE filter â€” the pipeline working dataset, used for junction analysis

In [ ]:
if os.path.exists(WEGVAKKEN_ROT_PATH) and os.path.exists(WEGVAKKEN_BST_PATH):
    # Fast path: preprocessed files from scripts/preprocess_nwb.py
    print("Loading preprocessed Rotterdam files (fast path) ...")
    wegvakken_rot = gpd.read_file(WEGVAKKEN_ROT_PATH)  # all Rotterdam, no filter
    wegvakken     = gpd.read_file(WEGVAKKEN_BST_PATH)  # WEGBEHSRT=G + BST filtered
else:
    # Slow fallback: load from national file
    print("Preprocessed files not found â€” loading national file (slow, ~30â€“60 s) ...")
    print("  Tip: run `python scripts/preprocess_nwb.py` once to avoid this in future sessions.")
    wegvakken_nl  = gpd.read_file(WEGVAKKEN_NL_PATH)
    wegvakken_rot = wegvakken_nl[wegvakken_nl["GME_NAAM"] == "Rotterdam"].copy()
    del wegvakken_nl
    wegvakken = wegvakken_rot[wegvakken_rot["WEGBEHSRT"] == WEGBEHSRT_GEMEENTE].copy()
    wegvakken = wegvakken[wegvakken["BST_CODE"].isin(INCLUDE_BST)].copy()

# Gemeente-only subset without BST filter â€” broader view for FOW/carriageway analysis.
# BST_CODE can hide FOW=2 roads that happen to have certain road type codes.
wegvakken_gemeente = wegvakken_rot[wegvakken_rot["WEGBEHSRT"] == WEGBEHSRT_GEMEENTE].copy()

print(f"wegvakken_rot      (all Rotterdam):         {len(wegvakken_rot):,} segments")
print(f"wegvakken_gemeente (WEGBEHSRT=G, no BST):   {len(wegvakken_gemeente):,} segments")
print(f"wegvakken          (WEGBEHSRT=G + BST):     {len(wegvakken):,} segments")

In [ ]:
# Load the processed intersections (junctions with street_count >= 3)
intersections = gpd.read_file(INTERSECTIONS_PATH).set_index("JTE_ID")
print(f"Intersections (junctions â‰¥3 streets): {len(intersections):,}")

## 2. How many junctions are affected by dual carriageways?

First, understand the scale of the problem: how many of our intersections
have at least one connected FOW=2 segment?

In [ ]:
# Diagnostic: FOW distribution at each filter level.
# NWB stores FOW as a string column ("2", "3", ...) â€” normalise to int for comparisons.
# Using wegvakken_gemeente (no BST filter) so BST_CODE cannot hide FOW=2 roads.

fow_labels = {1: "Motorway", 2: "Multiple carriageway", 3: "Single carriageway",
              4: "Roundabout", 6: "Sliproad", 7: "Other"}

for label, df in [("all Rotterdam (no filter)", wegvakken_rot),
                  ("WEGBEHSRT=G (no BST)",      wegvakken_gemeente),
                  ("WEGBEHSRT=G + BST",          wegvakken)]:
    # Normalise FOW to int so dict lookup and comparisons work regardless of storage type
    fow_series = pd.to_numeric(df["FOW"], errors="coerce").dropna().astype(int)
    dist = fow_series.value_counts().sort_index()
    n = len(df)
    print(f"\nFOW distribution â€” {label} ({n:,} segments):")
    for fow_val, count in dist.items():
        print(f"  FOW={fow_val} ({fow_labels.get(fow_val, 'Unknown')}): {count:,} ({count/n*100:.1f}%)")

# Summary: is FOW=2 present in the pipeline working dataset?
fow_bst = pd.to_numeric(wegvakken["FOW"], errors="coerce").astype(int)
if (fow_bst == 2).sum() == 0:
    print("\nNOTE: FOW=2 is absent from the BST-filtered pipeline dataset.")
    print("  Use wegvakken_gemeente for FOW-based analysis (Approaches Aâ€“C).")
    print("  Approaches D and E (proximity/graph) work on the pipeline dataset directly.")
else:
    n_fow2 = (fow_bst == 2).sum()
    print(f"\nFOW=2 present in pipeline dataset: {n_fow2:,} segments â€” FOW-based approaches will work.")

In [ ]:
# Map 2: FOW=3 roads that are suspiciously close to another FOW=3 road.
# These are candidates for being dual carriageways mislabelled or simply
# the FOW=3 portions of a physically separated road system.
#
# Method: compute the midpoint of each FOW=3 segment, then use a KD-tree to
# find all pairs of FOW=3 midpoints within a given distance threshold.
# Segments involved in such pairs are flagged as "apparent dual carriageway".

from scipy.spatial import cKDTree

CLOSE_THRESHOLD = 25  # metres â€” segments with a parallel neighbour within this distance

# Normalise FOW to int â€” NWB stores it as a string, which would break comparisons
fow_gemeente = pd.to_numeric(wegvakken_gemeente["FOW"], errors="coerce").astype("Int64")
wvk_fow2     = wegvakken_gemeente[fow_gemeente == 2]  # confirmed dual carriageways

fow3_seg = wegvakken_gemeente[fow_gemeente == 3].copy().reset_index(drop=True)

# Use midpoint of each segment as a representative location
midpoints = fow3_seg.geometry.centroid
coords = np.array([(p.x, p.y) for p in midpoints])

# Find all pairs of midpoints within the threshold distance
tree = cKDTree(coords)
pairs = tree.query_pairs(r=CLOSE_THRESHOLD)  # returns set of (i, j) index pairs

# Collect all segment indices that participate in at least one close pair
close_indices = set()
for i, j in pairs:
    close_indices.add(i)
    close_indices.add(j)

fow3_close = fow3_seg.iloc[sorted(close_indices)]   # FOW=3 with a close neighbour
fow3_far   = fow3_seg.iloc[[i for i in range(len(fow3_seg)) if i not in close_indices]]

print(f"FOW=3 segments with a neighbour within {CLOSE_THRESHOLD}m: {len(fow3_close):,} "
      f"({len(fow3_close)/len(fow3_seg)*100:.1f}%)")
print(f"FOW=3 segments without a close neighbour:                  {len(fow3_far):,}")

fig, ax = plt.subplots(figsize=(12, 12))

# Draw unambiguous FOW=3 roads in the background
fow3_far.plot(ax=ax,   color="steelblue",  linewidth=0.6, alpha=0.3, label="FOW=3 isolated")

# Draw candidate apparent dual carriageways on top
fow3_close.plot(ax=ax, color="darkorange", linewidth=1.5, alpha=0.85,
                label=f"FOW=3 with neighbour <{CLOSE_THRESHOLD}m (apparent dual carriageway?)")

# Overlay confirmed FOW=2 for comparison
wvk_fow2.plot(ax=ax, color="tomato", linewidth=1.5, alpha=0.7, label="FOW=2 (confirmed dual carriageway)")

# Overlay intersections
intersections.plot(ax=ax, color="black", markersize=2, zorder=5, alpha=0.5, label="Intersections")

ax.legend(fontsize=9)
ax.set_title(f"Apparent dual carriageways: FOW=3 segments with a parallel neighbour within {CLOSE_THRESHOLD}m\n"
             "Orange = likely dual carriageway but labelled FOW=3; Red = confirmed FOW=2")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Map 1: FOW=2 (dual carriageway) roads in Rotterdam.
# Using wegvakken_gemeente so BST_CODE filter doesn't hide these roads.

fow_gemeente = pd.to_numeric(wegvakken_gemeente["FOW"], errors="coerce").astype("Int64")

wvk_fow2 = wegvakken_gemeente[fow_gemeente == 2]
wvk_fow3 = wegvakken_gemeente[fow_gemeente == 3]
wvk_other = wegvakken_gemeente[~fow_gemeente.isin([2, 3])]

fig, ax = plt.subplots(figsize=(12, 12))

# Draw background roads first, then highlight FOW=2 on top
wvk_other.plot(ax=ax, color="lightgray",  linewidth=0.5, alpha=0.5, label="Other FOW")
wvk_fow3.plot(ax=ax,  color="steelblue",  linewidth=0.7, alpha=0.4, label="FOW=3 (single carriageway)")
wvk_fow2.plot(ax=ax,  color="tomato",     linewidth=1.8, alpha=0.9, label=f"FOW=2 (dual carriageway, n={len(wvk_fow2):,})")

# Overlay pipeline intersections for reference
intersections.plot(ax=ax, color="black", markersize=2, zorder=5, alpha=0.6, label="Intersections (pipeline)")

ax.legend(fontsize=9)
ax.set_title("FOW=2 dual carriageway roads â€” Rotterdam, WEGBEHSRT=G (no BST filter)\n"
             "Red = dual carriageway segments whose junctions may be split duplicates")
ax.set_axis_off()
plt.tight_layout()
plt.show()

print(f"FOW=2 segments in gemeente dataset: {len(wvk_fow2):,}")
print(f"FOW=3 segments in gemeente dataset: {len(wvk_fow3):,}")

In [ ]:
# For each junction (JTE_ID), check whether any connected segment has FOW=2.
#
# Two bugs fixed compared to a naive approach:
#   1. NWB stores FOW as a string ("2", "3") â€” must convert to numeric before comparing.
#   2. The BST-filtered `wegvakken` contains ZERO FOW=2 segments (confirmed by the diagnostic
#      cell above). We must use `wegvakken_gemeente` (no BST filter) so we can actually
#      find FOW=2 connections. Our intersections are still the BST-pipeline junctions.

# Normalise FOW to int in wegvakken_gemeente (string â†’ numeric)
wvk_g = wegvakken_gemeente[["JTE_ID_BEG", "JTE_ID_END", "FOW"]].copy()
wvk_g["FOW_int"] = pd.to_numeric(wvk_g["FOW"], errors="coerce")

# Build long-form table: one row per (junction, segment) connection
jte_fow = pd.concat([
    wvk_g[["JTE_ID_BEG", "FOW_int"]].rename(columns={"JTE_ID_BEG": "JTE_ID"}),
    wvk_g[["JTE_ID_END", "FOW_int"]].rename(columns={"JTE_ID_END": "JTE_ID"}),
], ignore_index=True)

# For each junction: does any connected segment (gemeente, no BST filter) have FOW=2?
has_fow2 = jte_fow.groupby("JTE_ID")["FOW_int"].apply(lambda x: (x == 2).any())

# Match to our pipeline intersections (BST-derived junctions)
inter_ids = intersections.index
affected  = has_fow2.reindex(inter_ids).fillna(False)

print(f"Intersections with â‰¥1 FOW=2 connected segment: {affected.sum():,} / {len(inter_ids):,} ({affected.mean()*100:.1f}%)")
print(f"Intersections with NO FOW=2 connection:        {(~affected).sum():,}")

### What are FOW=2 segments connected to?

At each junction endpoint of a FOW=2 segment, what other segment types appear?
This tells us whether FOW=2 dual carriageway segments mainly connect to other FOW=2 roads,
or whether they terminate into ordinary FOW=3 street junctions.

We also show a few zoomed-in junction examples to make the structure visible on a map.

In [ ]:
# Investigate what FOW=2 (dual carriageway) segments are actually connected to.
# At each junction endpoint of a FOW=2 segment, what other FOW and BST_CODE values appear?
# This reveals whether FOW=2 roads mainly link to other FOW=2 roads, or to ordinary FOW=3 streets.

# Work on wegvakken_gemeente (no BST filter) so we see all road types connected to FOW=2.
wvk_g2 = wegvakken_gemeente[["JTE_ID_BEG", "JTE_ID_END", "FOW", "BST_CODE", "STT_NAAM"]].copy()
wvk_g2["FOW_int"] = pd.to_numeric(wvk_g2["FOW"], errors="coerce")

# Identify all FOW=2 segments and collect their junction IDs
fow2_mask    = wvk_g2["FOW_int"] == 2
fow2_segs_g  = wvk_g2[fow2_mask]
fow2_jte_ids = set(fow2_segs_g["JTE_ID_BEG"].tolist() + fow2_segs_g["JTE_ID_END"].tolist())

print(f"FOW=2 segments (gemeente, no BST filter): {fow2_mask.sum():,}")
print(f"Unique junction endpoints of FOW=2 segments: {len(fow2_jte_ids):,}")

# Build long-form: one row per (junction, connected segment)
jte_long = pd.concat([
    wvk_g2.rename(columns={"JTE_ID_BEG": "JTE_ID"})[["JTE_ID", "FOW_int", "BST_CODE"]],
    wvk_g2.rename(columns={"JTE_ID_END": "JTE_ID"})[["JTE_ID", "FOW_int", "BST_CODE"]],
], ignore_index=True)

# Keep only rows where the junction is a FOW=2 endpoint
at_fow2_jte = jte_long[jte_long["JTE_ID"].isin(fow2_jte_ids)]

print(f"\nAll segments touching a FOW=2 junction endpoint: {len(at_fow2_jte):,} connections")

# FOW distribution at those junctions â€” do FOW=2 junctions mostly connect to other FOW=2 or to FOW=3?
fow_dist = pd.to_numeric(at_fow2_jte["FOW_int"], errors="coerce").value_counts().sort_index()
total_conn = len(at_fow2_jte)
fow_labels = {1: "Motorway", 2: "Dual carriageway", 3: "Single carriageway",
              4: "Roundabout", 6: "Sliproad", 7: "Other"}
print("\nFOW distribution at junctions touched by FOW=2 segments:")
for fow_val, count in fow_dist.items():
    label = fow_labels.get(int(fow_val), "Unknown")
    print(f"  FOW={int(fow_val)} ({label}): {count:,} ({count/total_conn*100:.1f}% of connections)")

# BST_CODE distribution â€” what road types meet at FOW=2 junctions?
bst_dist = at_fow2_jte["BST_CODE"].value_counts()
print("\nBST_CODE distribution at junctions touched by FOW=2 segments:")
for bst, count in bst_dist.items():
    print(f"  {bst}: {count:,} ({count/total_conn*100:.1f}%)")

In [ ]:
# Zoomed maps: pick a few junctions where FOW=2 meets FOW=3 and plot them at street level.
# This shows visually what the dual-carriageway / single-carriageway transition looks like.
#
# You can override SPECIFIC_JTE_IDS with junction IDs from your own inspection.
# If left empty, the cell auto-picks junctions at FOW=2/FOW=3 transition points.

SPECIFIC_JTE_IDS = []   # <- paste your own JTE_IDs here, e.g. [123456, 654321]
ZOOM_RADIUS      = 150  # metres around each junction to show
N_AUTO_EXAMPLES  = 6    # number of junctions to auto-pick (used when SPECIFIC_JTE_IDS is empty)

if SPECIFIC_JTE_IDS:
    # Use the user-specified junctions
    example_jte_ids = SPECIFIC_JTE_IDS
    print(f"Using {len(example_jte_ids)} user-specified junction IDs.")
else:
    # Auto-select: find junctions where FOW=2 and FOW=3 BOTH connect â€” the interesting boundary.
    # These are likely where a dual carriageway meets a normal street intersection.
    fow2_at_jte = at_fow2_jte.groupby("JTE_ID")["FOW_int"].apply(set)
    transition_jte = fow2_at_jte[
        fow2_at_jte.apply(lambda s: 2 in s and 3 in s)  # junction has both FOW=2 and FOW=3
    ].index.tolist()
    print(f"Junctions where FOW=2 and FOW=3 both connect (transition points): {len(transition_jte):,}")

    # Of those, prefer junctions that ARE in our pipeline intersections set
    in_pipeline = [jid for jid in transition_jte if jid in intersections.index]
    not_in_pipeline = [jid for jid in transition_jte if jid not in intersections.index]
    print(f"  Of which in pipeline intersections: {len(in_pipeline):,}")
    print(f"  Of which NOT in pipeline (dead-end or degree-2): {len(not_in_pipeline):,}")

    # Sample evenly from Rotterdam to get geographic variety (sort by x-coordinate)
    sample_pool = in_pipeline if len(in_pipeline) >= N_AUTO_EXAMPLES else transition_jte
    step = max(1, len(sample_pool) // N_AUTO_EXAMPLES)
    example_jte_ids = sample_pool[::step][:N_AUTO_EXAMPLES]
    print(f"\nAuto-selected {len(example_jte_ids)} example junctions (spatially spread).")

# --- Plot zoomed maps ---
n_cols = 3
n_rows = (len(example_jte_ids) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

for ax in axes[len(example_jte_ids):]:
    ax.set_visible(False)  # hide unused subplots

for i, jte_id in enumerate(example_jte_ids):
    ax = axes[i]

    # Find the junction's location â€” may be in intersections index or in the wegvakken endpoints
    if jte_id in intersections.index:
        centre_pt = intersections.loc[jte_id, "geometry"]
        in_pipe   = True
    else:
        # Derive location from a wegvak endpoint
        hit = wegvakken_gemeente[
            (wegvakken_gemeente["JTE_ID_BEG"] == jte_id) |
            (wegvakken_gemeente["JTE_ID_END"] == jte_id)
        ]
        if hit.empty:
            ax.set_title(f"JTE_ID {jte_id}\n(not found)")
            ax.set_axis_off()
            continue
        row0 = hit.iloc[0]
        if row0["JTE_ID_BEG"] == jte_id:
            centre_pt = Point(row0.geometry.coords[0])
        else:
            centre_pt = Point(row0.geometry.coords[-1])
        in_pipe = False

    # Buffer around the junction to get local context
    buf = centre_pt.buffer(ZOOM_RADIUS)

    # Clip wegvakken_gemeente to the buffer for context lines
    local_all  = wegvakken_gemeente[wegvakken_gemeente.geometry.intersects(buf)].copy()
    local_all["FOW_int"] = pd.to_numeric(local_all["FOW"], errors="coerce")

    # Plot each FOW class in a distinct colour
    style = {
        2: ("tomato",     2.5, 0.9, "FOW=2 dual"),
        3: ("steelblue",  1.2, 0.6, "FOW=3 single"),
        4: ("limegreen",  1.5, 0.8, "FOW=4 roundabout"),
        6: ("orchid",     1.2, 0.6, "FOW=6 sliproad"),
    }
    for fow_val, (color, lw, alpha, _) in style.items():
        subset = local_all[local_all["FOW_int"] == fow_val]
        if not subset.empty:
            subset.plot(ax=ax, color=color, linewidth=lw, alpha=alpha)

    # Draw remaining FOW values in grey
    other = local_all[~local_all["FOW_int"].isin(style.keys())]
    if not other.empty:
        other.plot(ax=ax, color="lightgray", linewidth=0.8, alpha=0.5)

    # Mark nearby pipeline intersections as small dots
    local_inter = intersections[intersections.geometry.within(buf)]
    if not local_inter.empty:
        local_inter.plot(ax=ax, color="black", markersize=8, zorder=5, alpha=0.7)

    # Highlight the focal junction with a star
    ax.plot(centre_pt.x, centre_pt.y,
            "r*" if in_pipe else "y*",
            markersize=18, zorder=10,
            label="focal junction (pipeline)" if in_pipe else "focal junction (not in pipeline)")

    # Set map extent to the buffer bounding box
    minx, miny, maxx, maxy = buf.bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(f"JTE_ID {jte_id}{'  âœ“ pipeline' if in_pipe else '  â€“ not pipeline'}", fontsize=9)
    ax.set_axis_off()

# Build a shared legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="tomato",    linewidth=2, label="FOW=2 dual carriageway"),
    Line2D([0], [0], color="steelblue", linewidth=2, label="FOW=3 single carriageway"),
    Line2D([0], [0], color="limegreen", linewidth=2, label="FOW=4 roundabout"),
    Line2D([0], [0], color="orchid",    linewidth=2, label="FOW=6 sliproad"),
    Line2D([0], [0], color="lightgray", linewidth=2, label="Other FOW"),
    Line2D([0], [0], marker="*", color="red",    markersize=12, linestyle="None", label="Focal junction (in pipeline)"),
    Line2D([0], [0], marker="*", color="yellow", markersize=12, linestyle="None", label="Focal junction (not in pipeline)"),
    Line2D([0], [0], marker="o", color="black",  markersize=6,  linestyle="None", label="Other pipeline intersections"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4, fontsize=8,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle(f"Zoomed views of FOW=2/FOW=3 transition junctions (radius={ZOOM_RADIUS}m)\n"
             "Red = dual carriageway; Blue = single carriageway; Star = focal junction",
             fontsize=11)
plt.tight_layout()
plt.show()

## 3. Approach A — FOW=2 as a scope filter
> **Verdict: used** — combined with Approach D as the FOW=2 distance threshold signal.

Identify junctions that are candidates for disaggregation:
those with at least one FOW=2 connected segment and another nearby junction
also connected to FOW=2 segments.

In [ ]:
# Subset intersections that have at least one FOW=2 connection
fow2_junctions = intersections[affected]
print(f"Candidate junctions (FOW=2 connected): {len(fow2_junctions):,}")

# Quick map: FOW=2 affected vs not
fig, ax = plt.subplots(figsize=(10, 10))
intersections[~affected].plot(ax=ax, color="steelblue", markersize=2, alpha=0.5, label="No FOW=2")
fow2_junctions.plot(ax=ax, color="tomato", markersize=4, alpha=0.8, label="FOW=2 connected")
ax.set_title("Intersections with FOW=2 connected segment (Rotterdam)")
ax.legend()
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Approach B — RIJRICHTNG as a complementary signal
> **Verdict: not used** — too noisy. Many FOW=2 segments have RIJRICHTNG=B (bidirectional),
> and many non-FOW=2 segments have H/T. The signal is redundant with FOW and less reliable as a standalone filter.

Per NWB documentation: *"In geval van gescheiden rijbanen hebben de wegvakken
altijd een gedwongen rijrichting"* — forced direction (H or T, never B).

So `RIJRICHTNG` in {H, T} is an independent signal confirming separated carriageways.

In [ ]:
# Distribution of RIJRICHTNG in our filtered wegvakken
rij = wegvakken["RIJRICHTNG"].value_counts(dropna=False)
print("RIJRICHTNG distribution (Rotterdam, BST filtered):")
print(rij.to_frame().assign(pct=(rij / len(wegvakken) * 100).round(1)))

In [ ]:
# How well does RIJRICHTNG=H/T overlap with FOW=2?
# We expect FOW=2 segments to always be H or T, and FOW=3 segments to mostly be B.
cross = pd.crosstab(wegvakken["FOW"], wegvakken["RIJRICHTNG"], margins=True)
print("Cross-tabulation FOW Ã— RIJRICHTNG:")
print(cross)

## 5. Approach C — RPE_CODE to pair dual carriageway twins
> **Verdict: not used** — RPE_CODE is sparsely populated for municipal roads (which have no WEGNUMMER).
> Works for Rijk/Provincie roads but Rotterdam is dominated by gemeente roads, giving insufficient coverage.

Per NWB documentation, `RPE_CODE` (column: `POS_TV_WOL`) gives the position of
a carriageway relative to the road orientation line:
- `L` = left carriageway
- `R` = right carriageway

Two FOW=2 segments with the **same `WEGNUMMER`** (or `STT_NAAM`) but **opposite RPE_CODE**
are the two carriageways of the same physical road.

First, check if RPE_CODE is populated in our dataset.

In [ ]:
# Check if RPE_CODE column exists and how it's populated
rpe_col = "RPE_CODE" if "RPE_CODE" in wegvakken.columns else "POS_TV_WOL"
print(f"Column used: {rpe_col}")

if rpe_col in wegvakken.columns:
    rpe = wegvakken[rpe_col].value_counts(dropna=False)
    print(f"\nRPE_CODE distribution (Rotterdam, BST filtered):")
    print(rpe.to_frame().assign(pct=(rpe / len(wegvakken) * 100).round(1)))
    
    # How many FOW=2 segments have L or R?
    fow2_segs = wegvakken[wegvakken["FOW"] == 2]
    rpe_fow2 = fow2_segs[rpe_col].value_counts(dropna=False)
    print(f"\nRPE_CODE for FOW=2 segments only ({len(fow2_segs):,} segments):")
    print(rpe_fow2.to_frame().assign(pct=(rpe_fow2 / len(fow2_segs) * 100).round(1)))
else:
    print("Column not found. Available columns:", list(wegvakken.columns))

In [ ]:
# If RPE_CODE is populated: try to pair dual carriageway segments
# Pair criteria: same WEGNUMMER (or STT_NAAM) + opposite RPE_CODE + FOW=2

fow2_segs = wegvakken[wegvakken["FOW"] == 2].copy()

# For roads with a wegnummer â€” Rijk/Provincie roads
has_wegnr = fow2_segs["WEGNUMMER"].notna() & (fow2_segs["WEGNUMMER"] != "")
print(f"FOW=2 segments with WEGNUMMER: {has_wegnr.sum():,} / {len(fow2_segs):,}")

# For roads without wegnummer â€” municipal roads (use STT_NAAM)
print(f"FOW=2 segments without WEGNUMMER (need STT_NAAM): {(~has_wegnr).sum():,}")

In [ ]:
# Attempt pairing by WEGNUMMER + RPE_CODE
# For each WEGNUMMER, find L/R pairs

if rpe_col in wegvakken.columns:
    paired = (
        fow2_segs[fow2_segs["WEGNUMMER"].notna()]
        .groupby("WEGNUMMER")[rpe_col]
        .apply(set)
    )
    # A paired road has both L and R
    fully_paired = paired[paired.apply(lambda s: {"L", "R"}.issubset(s))]
    print(f"Road numbers with both L and R carriageways: {len(fully_paired):,}")
    print(fully_paired.head(10))

## 6. Approach D — Proximity clustering
> **Verdict: used** — core merge signal. Two thresholds are applied: 30 m for FOW=2-connected pairs
> (dual carriageway duplicates) and 6 m for all pairs (bayonet intersections).
> See `exp_eda_nwb_bayonet.ipynb` for threshold calibration.

Group all intersections within a distance threshold into a single logical intersection.
No attribute knowledge needed — purely geometry based.

The key question: what threshold captures true duplicates without merging
genuinely separate intersections?

In [ ]:
# For each intersection, find its nearest neighbour and record the distance
from scipy.spatial import cKDTree

# Extract coordinates
coords = np.array([(geom.x, geom.y) for geom in intersections.geometry])
tree   = cKDTree(coords)

# Query 2 nearest (first is self, second is nearest other)
dists, _ = tree.query(coords, k=2)
nn_dist  = dists[:, 1]  # distance to nearest neighbour

print("Nearest-neighbour distance distribution (metres):")
for pct in [25, 50, 75, 90, 95, 99]:
    print(f"  p{pct}: {np.percentile(nn_dist, pct):.1f} m")
print(f"  min: {nn_dist.min():.1f} m")
print(f"  max: {nn_dist.max():.1f} m")

In [ ]:
# Histogram of nearest-neighbour distances â€” look for a bimodal structure:
# a peak near 0â€“30m (duplicate junctions) and a peak at typical block spacing
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(nn_dist, bins=100, range=(0, 200), color="steelblue", edgecolor="white")
ax.axvline(20, color="tomato",  linestyle="--", label="20 m (NWB bajonet threshold)")
ax.axvline(50, color="orange",  linestyle="--", label="50 m (physical separation threshold)")
ax.set_xlabel("Distance to nearest junction (m)")
ax.set_ylabel("Count")
ax.set_title("Nearest-neighbour distance between intersections â€” Rotterdam")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Test a few thresholds: how many junction pairs fall within them?
for threshold in [10, 20, 30, 50, 75]:
    n_close = (nn_dist < threshold).sum()
    print(f"  Junctions with a neighbour < {threshold:3d} m: {n_close:4d} ({n_close/len(nn_dist)*100:.1f}%)")

## 7. Approach E — Graph topology contraction
> **Verdict: not used** — the length histogram of inter-intersection segments shows no clean bimodal structure.
> There is no obvious threshold separating internal connectors from real street segments,
> making this approach unreliable without manual tuning.

Collapse clusters of junctions connected by very short road segments
(likely internal connector segments within an intersection complex).

Step 1: find the length distribution of segments connecting two intersections.

In [ ]:
# Identify wegvakken that connect two junctions that are BOTH in our intersections set
inter_set = set(intersections.index)

# A segment connects two intersections if both endpoints are intersections
inter_connecting = wegvakken[
    wegvakken["JTE_ID_BEG"].isin(inter_set) &
    wegvakken["JTE_ID_END"].isin(inter_set)
].copy()

# Compute segment length in metres (CRS is RD New, so units are already metres)
inter_connecting["length_m"] = inter_connecting.geometry.length

print(f"Segments connecting two intersections: {len(inter_connecting):,}")
print("\nLength distribution (metres):")
for pct in [5, 10, 25, 50, 75, 90, 95]:
    print(f"  p{pct}: {inter_connecting['length_m'].quantile(pct/100):.1f} m")

In [ ]:
# Histogram of inter-intersection segment lengths
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(inter_connecting["length_m"], bins=100, range=(0, 150), color="steelblue", edgecolor="white")
for t, c in [(20, "tomato"), (50, "orange")]:
    ax.axvline(t, color=c, linestyle="--", label=f"{t} m threshold")
ax.set_xlabel("Segment length (m)")
ax.set_ylabel("Count")
ax.set_title("Length of segments connecting two intersections")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# How many junction pairs would be merged at different length thresholds?
for threshold in [10, 20, 30, 50, 75]:
    n_short = (inter_connecting["length_m"] < threshold).sum()
    print(f"  Inter-intersection segments < {threshold:3d} m: {n_short:4d}  (would trigger merging)")

## 8. Approach comparison — summary

| Approach | Used? | Reason |
|---|---|---|
| A — FOW=2 scope | Yes | Reliable indicator of dual carriageway digitisation artefacts |
| B — RIJRICHTNG | No | Poor overlap with FOW=2; too noisy as a standalone signal |
| C — RPE_CODE | No | Sparsely populated for gemeente roads; insufficient coverage |
| D — Proximity clustering | Yes | Geometry-only, robust; calibrated with two separate thresholds |
| E — Graph contraction | No | No natural length threshold visible in the data |

**Final strategy:** Apply A as a scope predicate and D as the distance merge signal,
with two thresholds: 30 m where at least one junction in the pair has a FOW=2 connection,
and 6 m for all other pairs (bayonet signal). Transitive chains are resolved into clusters
before merging. The code block below checks the overlap between the two signals.

In [ ]:
# Combine FOW=2 scope (Approach A) with proximity (Approach D):
# How many of the close junction pairs have at least one FOW=2 connection?

affected_ids  = set(inter_ids[affected])  # junctions with FOW=2 connection

short_segs = inter_connecting[inter_connecting["length_m"] < 30]
short_either_fow2 = short_segs[
    short_segs["JTE_ID_BEG"].isin(affected_ids) |
    short_segs["JTE_ID_END"].isin(affected_ids)
]

print(f"Short segments (<30m) connecting two intersections:       {len(short_segs):,}")
print(f"  of which at least one endpoint has FOW=2 connection:   {len(short_either_fow2):,} ({len(short_either_fow2)/max(len(short_segs),1)*100:.1f}%)")

## 9. Visual inspection â€” example cases

Pick a few candidate junction clusters and plot them with their connecting wegvakken
to visually verify whether they represent the same physical intersection.

In [ ]:
# Pick the N shortest inter-intersection segments for inspection
N_EXAMPLES = 6
examples = inter_connecting.nsmallest(N_EXAMPLES, "length_m")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (_, row) in enumerate(examples.iterrows()):
    ax = axes[i]
    jte_a, jte_b = row["JTE_ID_BEG"], row["JTE_ID_END"]
    
    # Buffer around this segment to show context
    buf = row.geometry.centroid.buffer(100)
    local_wvk = wegvakken[wegvakken.geometry.intersects(buf)]
    local_jte = intersections[intersections.geometry.within(buf)]
    
    # Colour wegvakken by FOW
    fow_colors = {1: "red", 2: "orange", 3: "steelblue", 4: "green", 6: "purple", 7: "gray"}
    for fow_val, color in fow_colors.items():
        subset = local_wvk[local_wvk["FOW"] == fow_val]
        if not subset.empty:
            subset.plot(ax=ax, color=color, linewidth=1.5, alpha=0.8)
    
    # Mark all local intersections
    local_jte.plot(ax=ax, color="black", markersize=15, zorder=5)
    
    # Highlight the two junctions of this short segment
    for jte_id in [jte_a, jte_b]:
        if jte_id in intersections.index:
            pt = intersections.loc[jte_id, "geometry"]
            ax.plot(pt.x, pt.y, "r*", markersize=15, zorder=6)
    
    ax.set_title(f"Segment length: {row['length_m']:.1f} m\nFOW={row['FOW']}, {row.get('STT_NAAM', '')}", fontsize=9)
    ax.set_axis_off()

plt.suptitle("Shortest inter-intersection segments â€” candidate duplicate junctions\n(red stars = endpoints, coloured lines = FOW)", fontsize=11)
plt.tight_layout()
plt.show()